# ML Tutor — Week 8: Unsupervised learning + Checkpoint 1
**Track:** Core ML · **Lab:** Lab 8 — Cluster patient phenotypes and check what the map shows

**Objectives this week:**
1. Apply k-means and hierarchical clustering and validate cluster quality. *(CO2, CO5)*
2. Reduce dimensions with PCA/t-SNE and interpret the projections. *(CO2)*
3. Phenotype patients by clustering and demonstrate mid-course mastery (Checkpoint 1). *(CO2, CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive ready to look for structure without labels. This week is about clustering, dimensions, and whether a map helps you see patterns that a table hides. Reference delta: make one clustering view and one PCA view required; t-SNE stays advanced-only.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. From Week 7, what made a model comparison fair?
2. What is one reason we should be cautious about reading too much into a single score?
3. Why does preprocessing still matter even when we are not doing supervised learning?


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic phenotype table with fictional features such as age_band, comorbidity_index, visit_count_6m, medication_count, utilization_score, lab_variability, adherence_score, and phenotype_hint. Teaching-only data with no PHI.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-08/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-08/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — k-means clustering
**Lane A · the general idea:** k-means assigns each point to the nearest centroid, then updates the centroids repeatedly until the grouping stabilizes. It is simple, fast, and sensitive to feature scale.

**Lane B · the same idea in pharmacy terms:** For patient phenotyping, k-means can group records with similar utilization, comorbidity, or lab patterns. The result is a teaching hypothesis about subtypes, not a diagnosis.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — k-means finds patient groups WITHOUT ever seeing labels
import numpy as np
from sklearn.cluster import KMeans

# columns: [age, n_medications] — two phenotypes hiding in plain sight (synthetic)
patients = np.array([[34, 1], [29, 2], [41, 1], [72, 9], [68, 8], [75, 11]])

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(patients)
print("cluster of each patient:", km.labels_)
print("cluster centers [age, n_meds]:", km.cluster_centers_.round(1))
# 0/1 are ARBITRARY group ids — k-means never saw a diagnosis. Naming clusters is YOUR job.


## 1. Prepare the data for clustering
Choose a feature set, scale it, and make sure the clustering input is not dominated by raw units.

**Checkpoint:** Checkpoint 1 verifies: the selected features are appropriate for clustering and the data are scaled before any clustering algorithm is applied.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Synthetic phenotype panel (no PHI) — inline teaching data.
phenotype_df = pd.DataFrame([
    {"age_band": 1, "comorbidity_index": 2, "visit_count_6m": 3, "medication_count": 4, "utilization_score": 0.4, "lab_variability": 0.2, "adherence_score": 0.9, "phenotype_hint": "low_util"},
    {"age_band": 3, "comorbidity_index": 6, "visit_count_6m": 9, "medication_count": 11, "utilization_score": 0.8, "lab_variability": 0.6, "adherence_score": 0.5, "phenotype_hint": "high_util"},
    {"age_band": 2, "comorbidity_index": 3, "visit_count_6m": 4, "medication_count": 5, "utilization_score": 0.5, "lab_variability": 0.3, "adherence_score": 0.8, "phenotype_hint": "low_util"},
    {"age_band": 4, "comorbidity_index": 7, "visit_count_6m": 10, "medication_count": 12, "utilization_score": 0.9, "lab_variability": 0.7, "adherence_score": 0.4, "phenotype_hint": "high_util"},
    {"age_band": 1, "comorbidity_index": 1, "visit_count_6m": 2, "medication_count": 3, "utilization_score": 0.3, "lab_variability": 0.1, "adherence_score": 0.95, "phenotype_hint": "low_util"},
    {"age_band": 3, "comorbidity_index": 5, "visit_count_6m": 8, "medication_count": 10, "utilization_score": 0.75, "lab_variability": 0.55, "adherence_score": 0.55, "phenotype_hint": "high_util"},
    {"age_band": 2, "comorbidity_index": 2, "visit_count_6m": 3, "medication_count": 4, "utilization_score": 0.35, "lab_variability": 0.2, "adherence_score": 0.85, "phenotype_hint": "low_util"},
    {"age_band": 4, "comorbidity_index": 8, "visit_count_6m": 11, "medication_count": 13, "utilization_score": 0.95, "lab_variability": 0.8, "adherence_score": 0.35, "phenotype_hint": "high_util"},
])

feature_cols = [
    "age_band",
    "comorbidity_index",
    "visit_count_6m",
    "medication_count",
    "utilization_score",
    "lab_variability",
    "adherence_score",
]

X = phenotype_df[feature_cols]

numeric_cols = ["comorbidity_index", "visit_count_6m", "medication_count", "utilization_score", "lab_variability", "adherence_score"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop",
)

# TODO: fit_transform the numeric columns and keep the result ready for clustering
X_scaled = ...


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Run k-means and inspect the grouping
Fit a small k-means model, examine cluster counts, and summarize one cluster in plain language.

**Checkpoint:** Checkpoint 2 verifies: k-means is fit on the scaled data, cluster labels are attached to the dataframe, and the student can describe at least one cluster pattern.


In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

kmeans = KMeans(n_clusters=..., random_state=7, n_init="auto")
labels = kmeans.fit_predict(X_scaled)

phenotype_df = phenotype_df.copy()
phenotype_df["cluster"] = labels

# TODO: print cluster counts and one simple cluster summary
print(pd.Series(labels).value_counts().sort_index())
...


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Reduce dimensions for visualization
Use PCA and, if available in the notebook, t-SNE to create a 2D view of the phenotype structure.

**Checkpoint:** Checkpoint 3 verifies: PCA returns two components, the visualization table is built, and the student can explain what PCA preserves and what it does not.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=7)
pca_coords = pca.fit_transform(X_scaled)

# TODO: create a small dataframe with PC1 and PC2 and attach cluster labels
viz_df = ...


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Interpret the phenotype map and prepare for Checkpoint 1
Write a short interpretation of the cluster structure, its limitations, and one question that should be checked before anyone over-trusts the grouping.

**Checkpoint:** Checkpoint 4 verifies: the student can explain the cluster structure in plain language, name one limitation, and articulate what should be validated before over-claiming.


In [ ]:
# TODO: write a concise interpretation summary
summary = """
...
"""

print(summary)


**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“Clustering gives the true labels.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: Checkpoint 1 reflection memo
Write a short reflection on the phenotype clustering workflow. Summarize what the clusters might mean, what the PCA/t-SNE views helped you see, and one reason the result still needs caution before anyone treats it as a final answer.

**Deliverable:** A one-page reflection memo plus a screenshot or note of the main cluster summary and visualization.


In [ ]:
# HW / challenge workspace — build it here

